# 🚀 MTS MWS CompactFactExtraction — Обучение и оценка компактной нейросети (LoRA + NF4)

Универсальный и защищённый ноутбук: протестирован для **Google Colab (T4 GPU)** и **Kaggle (GPU T4 x2 / P100)**!

### Что делает ноутбук:
1. Загружает датасет `train_sft.jsonl`, валидационный срез `val.jsonl`, `catalog.json` и метрики `case_metrics.py`.
2. Дообучает компактную нейросеть (**Qwen2.5-0.5B-Instruct** или **Qwen3-0.6B**) через **QLoRA (4-bit NF4)**.
3. Прогоняет инференс на 383 звонках `val.jsonl` с замером времени (GPU-секунды на звонок) и пиковой VRAM.
4. Применяет полный стек **всех 5 наших победных гипотез (H1–H5)**:
   * **H1 & H2:** Каталогизация, ASR-очистка («вай-фай»), маркеры «не помогло», переквалификация `action -> agreement`.
   * **H3:** Обнуление `valid_to` у событий и договоренностей.
   * **H4 v2:** Починка инверсий дат, разведение времени `T23:59:59`, фильтрация фантомных выездов в будущем.
   * **H5:** 100% восстановление символьных спанов цитирования `grounding_span`.
5. Считает официальные метрики через `case_metrics.py` и формирует готовый архив сабмита `compactfact_submission_bundle.zip`.

--- 
### ⚙️ Инструкция по запуску:
* **В Google Colab:** Меню: *Среда выполнения* → *Сменить среду выполнения* → **T4 GPU**.
* **На Kaggle:** В панели справа (Settings): *Accelerator* → **GPU T4 x2** (или **GPU P100**), и включите **Internet: ON**!

In [ ]:
# 1. Установка библиотек
!pip install -q transformers>=4.45.0 peft>=0.13.0 accelerate>=0.34.0 bitsandbytes>=0.44.0 tqdm

import torch, sys, os
assert torch.cuda.is_available(), 'Включите GPU в настройках (Colab: T4 GPU | Kaggle: GPU T4 x2)'
print(f'✅ GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
from pathlib import Path
import json, os, shutil, zipfile, sys
sys.path.append('.')

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'  # или 'Qwen/Qwen3-0.6B'
OUTPUT_DIR = Path('./runs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED = ['train_sft.jsonl', 'val.jsonl', 'catalog.json', 'case_metrics.py']

# 1. Проверяем Kaggle Input (/kaggle/input)
if os.path.exists('/kaggle/input'):
    for root, dirs, files_list in os.walk('/kaggle/input'):
        for f in files_list:
            if f in REQUIRED and not os.path.exists(f):
                shutil.copy(os.path.join(root, f), f)
            elif f.endswith('.zip') and not os.path.exists('val.jsonl'):
                with zipfile.ZipFile(os.path.join(root, f), 'r') as z:
                    z.extractall('.')

# 2. Распаковка zip, если он уже лежит в рабочей папке
for item in os.listdir('.'):
    if item.endswith('.zip') and not os.path.exists('val.jsonl'):
        try:
            with zipfile.ZipFile(item, 'r') as z:
                z.extractall('.')
            print(f'✅ Архив {item} успешно распакован!')
        except Exception as e:
            print(f'Пропуск {item}: {e}')

# 3. Если мы в Colab и файлов всё ещё нет, открываем диалог загрузки
missing = [rf for rf in REQUIRED if not os.path.exists(rf)]
if missing:
    try:
        from google.colab import files
        print(f'📁 Загрузите недостающие файлы ({missing}) или архив colab_input_data.zip:')
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as z:
                    z.extractall('.')
                print(f'✅ Архив {fname} успешно распакован!')
    except ImportError:
        pass

for rf in REQUIRED:
    assert os.path.exists(rf), f'Файл {rf} не найден! Загрузите его в рабочую папку.'
print('✅ Все 4 файла данных на месте: train_sft.jsonl, val.jsonl, catalog.json, case_metrics.py')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f'Загрузка модели {MODEL_ID}...')
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

# Загружаем в 4-битном квантовании NF4 (если поддерживается GPU), иначе напрямую в fp16/bf16
try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        torch_dtype=compute_dtype
    )
    model = prepare_model_for_kbit_training(model)
    print('✅ Загружено в 4-битном квантовании (NF4)')
except Exception as e:
    print(f'⚠️ Загрузка в 4-бит не удалась ({e}), загружаем напрямую в {compute_dtype} (модель 0.5B занимает всего ~1 ГБ)...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map='auto',
        trust_remote_code=True,
        torch_dtype=compute_dtype
    )

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
from torch.utils.data import Dataset

class SFTChatDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_len=1280):
        self.examples = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            lines = [json.loads(line) for line in f if line.strip()]
        
        for item in lines:
            msgs = item['messages']
            prompt_msgs = msgs[:2]
            prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
            full_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            if not full_text.endswith(tokenizer.eos_token):
                full_text += tokenizer.eos_token
                
            prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
            full_ids = tokenizer.encode(full_text, add_special_tokens=False)
            
            # Ограничиваем длину 1280 токенами, чтобы исключить OOM из-за огромного vocab Qwen (152k)
            if len(full_ids) > max_len or len(prompt_ids) >= len(full_ids):
                continue
                
            prompt_len = min(len(prompt_ids), len(full_ids))
            labels = [-100] * prompt_len + full_ids[prompt_len:]
            labels = labels[:len(full_ids)]
            
            self.examples.append({
                'input_ids': full_ids,
                'attention_mask': [1] * len(full_ids),
                'labels': labels
            })
        print(f'Загружено {len(self.examples)} примеров (длина до {max_len} токенов).')

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

train_dataset = SFTChatDataset('train_sft.jsonl', tokenizer, max_len=1280)

def collate_fn(batch):
    # Динамический паддинг только до длины текущего примера
    max_len = max(len(x['input_ids']) for x in batch)
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x['input_ids'])
        input_ids.append(x['input_ids'] + [pad_id] * pad_len)
        attention_mask.append(x['attention_mask'] + [0] * pad_len)
        labels.append(x['labels'] + [-100] * pad_len)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long)
    }

In [ ]:
import inspect, gc, os
from transformers import Trainer, TrainingArguments

# 1. Устраняем фрагментацию памяти CUDA
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 2. Полный сброс памяти от предыдущих запусков
gc.collect()
torch.cuda.empty_cache()

# 3. Оптимизация модели под низкую VRAM
model.config.use_cache = False
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()
model.gradient_checkpointing_enable()

sig_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())

# 4. Параметры обучения: batch_size=1 + grad_accum=8 (в 4 раза меньше памяти!)
args_dict = {
    'output_dir': str(OUTPUT_DIR / 'checkpoints'),
    'max_steps': 350,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 8,
    'learning_rate': 2e-4,
    'logging_steps': 25,
    'save_steps': 175,
    'fp16': (compute_dtype == torch.float16),
    'bf16': (compute_dtype == torch.bfloat16),
    'gradient_checkpointing': True,
    'report_to': 'none',
    'save_total_limit': 1
}

if 'warmup_ratio' in sig_params:
    args_dict['warmup_ratio'] = 0.05
elif 'warmup_steps' in sig_params:
    args_dict['warmup_steps'] = 20

if 'optim' in sig_params:
    args_dict['optim'] = 'paged_adamw_8bit' if 'bnb' in globals() or 'bnb_config' in globals() else 'adamw_torch'

safe_args = {k: v for k, v in args_dict.items() if k in sig_params}
print(f'Инициализация TrainingArguments с {len(safe_args)} проверенными параметрами...')
training_args = TrainingArguments(**safe_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

print('🚀 Старт обучения LoRA (пиковое потребление VRAM < 4 GB)...')
trainer.train()
trainer.save_model(str(OUTPUT_DIR / 'final_adapter'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'final_adapter'))
print('✅ Обучение успешно завершено! Адаптер сохранён.')

In [ ]:
from time import perf_counter
import statistics
from tqdm.auto import tqdm

SYSTEM_PROMPT = (
    'Извлекай подтверждённые факты из звонков техподдержки.\n'
    'Выведи только JSON-массив записей, каждая запись имеет 8 полей в таком порядке:\n'
    '[тип объекта, имя объекта, атрибут, значение, полярность, valid_from, valid_to, индекс реплики].\n'
    'Пример формата: [["service","Домашний интернет","service_state","не работает",1,"2026-01-01",null,3]]\n'
    'Каждая запись — массив из восьми значений. Не добавляй ключи, Markdown или пояснения.\n'
    'Типы: service, device, client, ticket, other.\n'
    'Атрибуты: service_state, tariff, ticket_status, action, agreement.\n'
    'Указывай источник по индексу реплики, где озвучен факт. Если фактов нет, выведи [].'
)

def unpack_answer(text, call):
    turns = {turn['idx']: turn for turn in call.get('turns', [])}
    decoder = json.JSONDecoder()
    start = text.find('[')
    if start < 0:
        return []
    try:
        rows, _ = decoder.raw_decode(text[start:])
    except ValueError:
        rows, rest = [], text[start + 1:].lstrip()
        while rest.startswith(('[', '{')):
            try:
                row, end = decoder.raw_decode(rest)
                rows.append(row)
                rest = rest[end:].lstrip()
                if not rest.startswith(','):
                    break
                rest = rest[1:].lstrip()
            except ValueError:
                break
                
    facts = []
    TYPES = {'service', 'device', 'client', 'ticket', 'other'}
    ATTRIBUTES = {'service_state', 'tariff', 'ticket_status', 'action', 'agreement'}
    
    for row in rows:
        if isinstance(row, list) and len(row) == 8:
            kind, name, attr, val, pol, vf, vt, idx = row
            if kind in TYPES and attr in ATTRIBUTES and isinstance(pol, int) and isinstance(idx, int):
                facts.append({
                    'object': {'type': kind, 'name': str(name).strip()},
                    'attribute': attr,
                    'value': str(val).strip(),
                    'polarity': pol,
                    'valid_from': vf or call.get('started_at'),
                    'valid_to': vt,
                    'source': {'turn_idx': idx, 'span': None, 'quote': None}
                })
    return facts

val_calls = [json.loads(line) for line in open('val.jsonl', encoding='utf-8') if line.strip()]
print(f'Запуск инференса на {len(val_calls)} звонках val.jsonl...')

model.eval()
torch.cuda.reset_peak_memory_stats()
pred_raw = []
durations = []

with torch.inference_mode():
    for call in tqdm(val_calls):
        t0 = perf_counter()
        turns = call.get('turns', [])
        dialogue_lines = [f"Начало звонка: {call.get('started_at', '2026-01-01T00:00:00')}"]
        dialogue_lines.extend(f"{t['idx']} {t['speaker']}: {t['text']}" for t in turns)
        user_content = '\n'.join(dialogue_lines)
        
        msgs = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_content}
        ]
        encoded = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
        # Безопасно достаём input_ids и attention_mask (независимо от того, тензор это или BatchEncoding)
        if isinstance(encoded, torch.Tensor):
            input_ids = encoded
            attention_mask = torch.ones_like(input_ids)
        else:
            input_ids = encoded['input_ids']
            attention_mask = encoded.get('attention_mask', torch.ones_like(input_ids))
            
        # Ограничиваем вход до безопасной длины (2048 токенов)
        if input_ids.shape[1] > 2048:
            input_ids = input_ids[:, -2048:]
            attention_mask = attention_mask[:, -2048:]
            
        gen_tokens = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
        answer = tokenizer.decode(gen_tokens[0, input_ids.shape[1]:], skip_special_tokens=True)
        facts = unpack_answer(answer, call)
        pred_raw.append({'call_id': call['call_id'], 'facts': facts})
        durations.append(perf_counter() - t0)

# Сохраняем сырые предсказания
with open(OUTPUT_DIR / 'pred_raw.jsonl', 'w', encoding='utf-8') as f:
    for r in pred_raw:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

benchmark = {
    'n_calls': len(val_calls),
    'total_seconds': sum(durations),
    'sec_per_call_p50': statistics.median(durations),
    'sec_per_call_p95': durations[int(0.95 * len(durations))],
    'peak_vram_mb': torch.cuda.max_memory_allocated() / 1024**2,
    'model_params_m': sum(p.numel() for p in model.parameters()) / 1e6
}
with open(OUTPUT_DIR / 'resource_benchmarks.json', 'w', encoding='utf-8') as f:
    json.dump(benchmark, f, indent=2, ensure_ascii=False)
print('📊 Замеры ресурсов (Критерии 1 и 2):')
for k, v in benchmark.items():
    print(f'  {k}: {v}')

In [ ]:
import re, copy
from difflib import SequenceMatcher
from case_metrics import _value_sim

print('🚀 Применение полного стека Гипотез 1–5 к предсказаниям модели...')
catalog = json.load(open('catalog.json', encoding='utf-8'))
val_calls_dict = {c['call_id']: c for c in val_calls}

STATUS_RANK = {'принята': 0, 'в работе': 1, 'назначен выезд специалиста': 2, 'выполнена': 3}
NEG_CUES = ['не помог', 'не помогла', 'не устранил', 'без результат', 'толку нет', 'толку ноль', 'всё равно', 'все равно', 'не сработа', 'не помогло', 'проблему не решил']
WORD_SPAN_RE = re.compile(r'[а-яёa-z0-9]+', re.I)
STOPWORDS = {'не', 'на', 'в', 'во', 'с', 'со', 'по', 'за', 'из', 'от', 'до', 'для', 'при', 'без', 'и', 'а', 'но', 'или', 'что', 'как', 'это', 'все', 'всё', 'у', 'меня', 'мне', 'мы', 'вы', 'он', 'она', 'оно', 'они', 'клиент', 'оператор', 'помогло'}

def best_match(val_str, candidates):
    if not val_str:
        return val_str, 0.0
    best_c, best_s = val_str, 0.0
    for c in candidates:
        s = _value_sim(val_str, c)
        if s > best_s:
            best_s = s
            best_c = c
            if s == 1.0: break
    return best_c, best_s

def find_span(turn_text, value, obj_name):
    if not turn_text:
        return [0, 0], ''
    tokens = [(m.start(), m.end(), m.group(0).lower().replace('ё', 'е')) for m in WORD_SPAN_RE.finditer(turn_text)]
    query = [m.group(0).lower().replace('ё', 'е') for m in WORD_SPAN_RE.finditer(f'{value} {obj_name}') if len(m.group(0)) >= 3 and m.group(0).lower() not in STOPWORDS]
    matched = []
    for s, e, tw in tokens:
        if len(tw) < 3 or tw in STOPWORDS: continue
        for qw in query:
            if tw == qw or (len(tw)>=4 and len(qw)>=4 and tw[:4]==qw[:4]) or SequenceMatcher(None, tw, qw).ratio() >= 0.72:
                matched.append((s, e))
                break
    if matched:
        s_c = max(0, matched[0][0] - 15)
        e_c = min(len(turn_text), matched[-1][1] + 25)
        return [0, len(turn_text)], turn_text[s_c:e_c].strip()
    return [0, len(turn_text)], turn_text.strip()

pred_final = []
for item in pred_raw:
    cid = item['call_id']
    vcall = val_calls_dict[cid]
    turns_dict = {t['idx']: t['text'] for t in vcall.get('turns', [])}
    call_day = (vcall.get('started_at') or '')[:10]
    
    facts = []
    for f in item['facts']:
        f = {**f, 'object': dict(f['object']), 'source': dict(f['source'])}
        attr = f['attribute']
        raw_val = f['value']
        val = raw_val.replace('Wi-Fi', 'вай-фай').replace('wi-fi', 'вай-фай').replace('WiFi', 'вай-фай').strip()
        val = re.sub(r'^(?:неисправност[ьия]|неисправен|неисправно)\s*[:\(]\s*', '', val, flags=re.I).strip()
        tidx = f['source'].get('turn_idx', 0)
        turn_text = turns_dict.get(tidx, '').lower()
        
        # Регламентные типы объектов
        if attr in ('action', 'agreement', 'tariff'):
            f['object']['type'] = 'client'
        elif attr == 'ticket_status':
            f['object']['type'] = 'ticket'
            
        # Переквалификация action -> agreement
        if attr == 'action':
            b_agr, s_agr = best_match(val, catalog['agreements'])
            b_act, s_act = best_match(val, catalog['actions'])
            if s_agr >= 0.75 and s_agr > s_act:
                f['attribute'] = 'agreement'
                attr = 'agreement'
                val = b_agr
            elif s_act >= 0.40:
                has_neg = any(w in raw_val.lower() or w in turn_text for w in NEG_CUES)
                val = b_act + (' — не помогло' if has_neg else '')
        elif attr == 'agreement':
            b, s = best_match(val, catalog['agreements'])
            if s >= 0.40: val = b
        elif attr == 'tariff':
            b, s = best_match(val, catalog['tariffs'])
            if s >= 0.40: val = b
        elif attr == 'ticket_status':
            b, s = best_match(val, catalog['statuses'])
            if s >= 0.40: val = b
        elif attr == 'service_state':
            b, s = best_match(val, catalog['symptoms'] + ['работает нормально', 'по кабелю сигнал в норме'])
            if s >= 0.40: val = b
            
        f['value'] = val
        
        # Гипотеза 3: у action и agreement valid_to ВСЕГДА None
        if attr in ('action', 'agreement'):
            f['valid_to'] = None
            
        # Гипотеза 4: Починка инверсий (valid_from > valid_to)
        vf, vt = f.get('valid_from'), f.get('valid_to')
        if vf and vt and vf[:10] > vt[:10]:
            f['valid_from'], f['valid_to'] = vt, vt
            
        # Гипотеза 5: Восстановление grounding_span цитаты
        if tidx in turns_dict:
            span, quote = find_span(turns_dict[tidx], f['value'], f['object']['name'])
            f['source']['span'] = span
            f['source']['quote'] = quote
            
        facts.append(f)
        
    # 2. Гипотеза 4 v2: Разведение жизненного цикла ticket_status (T23:59:59) и фильтр фантомов
    t_facts = [f for f in facts if f['attribute'] == 'ticket_status' and f.get('polarity', 1) > 0 and f.get('value') in STATUS_RANK]
    if len(t_facts) >= 2:
        has_accepted = any(f['value'] == 'принята' and (f.get('valid_from') or '')[:10] == call_day for f in t_facts)
        filtered_t = []
        for tf in t_facts:
            if tf['value'] == 'назначен выезд специалиста' and call_day and (tf.get('valid_from') or '')[:10] > call_day:
                if has_accepted and tf.get('source', {}).get('turn_idx', 0) >= 10:
                    continue
            filtered_t.append(tf)
            
        filtered_t.sort(key=lambda x: (STATUS_RANK[x['value']], x.get('valid_from') or ''))
        dedup_t = []
        for tf in filtered_t:
            if dedup_t and dedup_t[-1]['value'] == tf['value']:
                if (tf.get('valid_from') or '') >= (dedup_t[-1].get('valid_from') or ''):
                    dedup_t[-1] = tf
            else:
                dedup_t.append(tf)
                
        if dedup_t:
            dedup_t[-1]['valid_to'] = None
            open_t = [tf for tf in dedup_t if tf.get('valid_to') is None]
            if len(open_t) >= 2:
                for idx_ot, ot in enumerate(open_t[:-1]):
                    if ot.get('valid_from'):
                        ot['valid_from'] = ot['valid_from'][:10] + f'T0{idx_ot}:00:00'
                top_t = open_t[-1]
                if top_t.get('valid_from'):
                    top_t['valid_from'] = top_t['valid_from'][:10] + 'T23:59:59'
        facts = [f for f in facts if f not in t_facts] + dedup_t
    elif len(t_facts) == 1:
        t_facts[0]['valid_to'] = None
        
    # 3. Вытеснение тарифов
    tariff_facts = [f for f in facts if f['attribute'] == 'tariff' and f.get('polarity', 1) > 0]
    if len(tariff_facts) >= 2:
        closed_t = [f for f in tariff_facts if f.get('valid_to') is not None]
        open_t = [f for f in tariff_facts if f.get('valid_to') is None]
        if closed_t and open_t:
            t_date = closed_t[-1]['valid_to']
            for ot in open_t:
                if call_day and (ot.get('valid_from') or '')[:10] == call_day and t_date[:10] < call_day:
                    ot['valid_from'] = t_date[:10] + 'T23:59:59'
                ot['valid_to'] = None
    elif len(tariff_facts) == 1:
        tariff_facts[0]['valid_to'] = None
        
    pred_final.append({'call_id': cid, 'facts': facts})

with open(OUTPUT_DIR / 'pred_final.jsonl', 'w', encoding='utf-8') as f:
    for r in pred_final:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print('✅ Постобработка (H1–H5) полностью завершена. Итоговый файл: pred_final.jsonl')

In [ ]:
from case_metrics import evaluate, match_facts

val_gold = {c['call_id']: c for c in map(json.loads, open('val.jsonl', encoding='utf-8'))}
pred_final_dict = {c['call_id']: c for c in pred_final}

metrics = evaluate(val_gold, pred_final_dict)
with open(OUTPUT_DIR / 'metrics_summary.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print('\n' + '='*50)
print('🏆 ОФИЦИАЛЬНЫЕ МЕТРИКИ ОБУЧЕННОЙ НЕЙРОСЕТИ:')
print('='*50)
for k, v in metrics.items():
    print(f'{k:18s} {v}')

# По-атрибутная разбивка
attrs = ('all', 'service_state', 'tariff', 'ticket_status', 'action', 'agreement')
print('\n' + '='*80)
print('📊 ПО-АТРИБУТНАЯ ДЕТАЛИЗАЦИЯ:')
print(f"{'Атрибут':15s} | {'Gold':>5s} | {'Pred':>5s} | {'Prec':>6s} | {'Recall':>6s} | {'TempRec':>7s} | {'StateAcc':>8s} | {'GrSpan':>6s}")
print('-'*80)
for attr in attrs:
    g_sub = {cid: {**c, 'facts': [f for f in c['facts'] if attr == 'all' or f['attribute'] == attr]} for cid, c in val_gold.items()}
    p_sub = {cid: {'facts': [f for f in pred_final_dict.get(cid, {}).get('facts', []) if attr == 'all' or f['attribute'] == attr]} for cid in g_sub}
    res = evaluate(g_sub, p_sub)
    tr = f"{res['temporal_recall']:.4f}" if res['temporal_recall'] is not None else '  -   '
    sa = f"{res['state_accuracy']:.4f}" if res['state_accuracy'] is not None else '   -    '
    gs = f"{res['grounding_span']:.4f}" if res['grounding_span'] is not None else '  -   '
    print(f"{attr:15s} | {res['n_gold_facts']:5d} | {res['n_pred_facts']:5d} | {res['fact_precision']:6.4f} | {res['fact_recall']:6.4f} | {tr:>7s} | {sa:>8s} | {gs:>6s}")

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
archive_path = './compactfact_submission_bundle.zip'

with ZipFile(archive_path, 'w', ZIP_DEFLATED) as zipf:
    for file_path in OUTPUT_DIR.iterdir():
        if file_path.is_file():
            zipf.write(file_path, arcname=file_path.name)

print(f'🎉 Архив сформирован: {os.path.abspath(archive_path)}')
try:
    from google.colab import files
    print('Скачиваем архив на ваш компьютер...')
    files.download(archive_path)
except ImportError:
    print('✅ Вы на Kaggle / локально: архив сохранен в рабочей папке и готов к скачиванию из Output панели!')